In [1]:
from B0_MODEL import UNET
from B2_UTILS import (load_checkpoint, save_checkpoint, get_loaders,
                      check_accuracy, save_predictions_as_imgs,
                      print_and_write)
import torch
from tqdm import tqdm  # makes your loops show a smart progress meter
import torch.nn as nn
import torch.optim as optim
import os
import json


In [2]:
# Train? Test training?
TRAIN = True
BATCH_1_TEST = True

# Set training epoch munbers, early stopping epoch num
training_epoch_num = 1000
early_stopping_patience = 5

In [3]:
# ====================== Hyperparameters etc. ========================
MAIN_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST'
MODEL_TYPES = ['secondary_strctures', 'key_atoms', 'residue_types']
MAP_DIRS_TYPES = [['map_sample'], ['map_sample', 'secondary_strctures'],
                  ['map_sample', 'secondary_strctures', 'key_atoms']]
# LOAD_MODEL = False

type_index = 0
model_type = MODEL_TYPES[type_index]
map_dirs = MAP_DIRS_TYPES[type_index]
in_channels = len(map_dirs)

data_path = os.path.join(MAIN_PATH, 'train_val_data')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 8  # batch_size vs GPU Memory: 1vs2877; 2vs3485; 4vs4668; 8vs7162

num_workers = 2
learning_rate = 1e-4
pin_memory = True
# IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_DEPTH = [64, 64, 64]


In [4]:
def initial_setting(model_type, data_path, device):
    """
    This function performs the initial setup before training process:
    1. Read the class weights of the UNET model,
    2. Read the saved epoch number,
    3. Generate paths of file that will be loaded to UNET model.
    """

    # Load the class weights of all model types
    weight_path = os.path.join(data_path, 'class_weight_for_training.txt')
    try:
        # Load the weight for training
        with open(weight_path, "r") as file:
            weight_all = json.load(file)

        if model_type not in weight_all.keys():
            print(f"Weight for {model_type} is not found.")
            exit()
        else:
            c_w = weight_all[model_type]
            class_weights = torch.FloatTensor(c_w).to(device)

            # Calculate the number of output channels based on the number of tags in model_type
            out_channels = len(c_w)

    except FileNotFoundError:
        print("Weight file does not exist.")
        exit()

    # Load the load_epoch_num data or initialize it if the file doesn't exist
    load_epoch_num_path = os.path.join(data_path, "epoch_num_path.txt")

    if not os.path.exists(load_epoch_num_path):
        load_epoch_num_all = {model_type: 0}
    else:
        with open(load_epoch_num_path, "r") as file:
            load_epoch_num_all = json.load(file)
            if model_type not in load_epoch_num_all.keys():
                load_epoch_num_all[model_type] = 0
    load_epoch_num = load_epoch_num_all[model_type]

    # Save the updated load_epoch_num_all data
    with open(load_epoch_num_path, "w") as file:
        json.dump(load_epoch_num_all, file)

    # Set file paths and make some directories when not existed
    train_map_dirs = [
        os.path.join(data_path, 'train', map_dir) for map_dir in map_dirs
    ]
    val_map_dirs = [
        os.path.join(data_path, 'val', map_dir) for map_dir in map_dirs
    ]
    train_model_dir = os.path.join(data_path, 'train', model_type)
    val_model_dir = os.path.join(data_path, 'val', model_type)

    load_model_path = os.path.join(data_path, 'checkpoint', model_type,
                               f'{model_type}.epoch{load_epoch_num}.pth.tar')

    evl_folder_path = os.path.join(data_path, 'evaluation', model_type)

    os.makedirs(os.path.dirname(load_model_path), exist_ok=True)
    os.makedirs(evl_folder_path, exist_ok=True)

    return class_weights, out_channels, load_epoch_num, train_map_dirs, val_map_dirs, train_model_dir, val_model_dir, load_model_path, evl_folder_path


In [5]:
def train_fn(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)
    total_loss = 0

    for batch_idx, (data, targets) in enumerate(loop):
        data, targets = data.to(device=DEVICE), targets.long().to(device=DEVICE)


        # forward
        with torch.cuda.amp.autocast():
            predictions = model(data)
            loss = loss_fn(predictions, targets)
            total_loss += loss.item()

        # backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # update tqdm loop
        loop.set_postfix(loss=loss.item())
    average_loss = total_loss / (batch_idx + 1)

    return round(average_loss, 4)

In [6]:
def batch_1_test(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)

    data, targets = next(iter(loader))
    data, targets = data.to(device=DEVICE), targets.long().to(device=DEVICE)
    print(data.shape)
    print(targets.shape)

    for epoch in range(500):
        # forward
        with torch.cuda.amp.autocast():
            predictions = model(data)
            print(predictions.shape)
            loss = loss_fn(predictions, targets)


        # backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # update tqdm loop
        loop.set_postfix(loss=loss.item())

In [7]:
# model = UNET(in_channels=in_channels, out_channels=out_channels).to(DEVICE)
# # loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
# loss_fn = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=learning_rate)
# scaler = torch.cuda.amp.GradScaler()

In [8]:

# train_loader, val_loader = get_loaders(train_map_dirs=train_map_dirs,
#                                         train_model_dir=train_model_dir,
#                                         val_map_dirs=val_map_dirs,
#                                         val_model_dir=val_model_dir,
#                                         batch_size=batch_size,
#                                         num_workers=num_workers,
#                                         pin_memory=pin_memory)

In [9]:
# train_fn(train_loader, model, optimizer, loss_fn, scaler)

In [10]:
class_weights, out_channels, load_epoch_num, train_map_dirs, val_map_dirs, train_model_dir, val_model_dir, load_model_path, evl_folder_path = initial_setting(model_type, data_path, DEVICE)

model = UNET(in_channels, out_channels).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scaler = torch.cuda.amp.GradScaler()

In [11]:
print(class_weights)
print(out_channels)
print(load_epoch_num)
print(train_map_dirs)
print(val_map_dirs)
print(train_model_dir)
print(val_model_dir)
print(load_model_path)
print(evl_folder_path)

tensor([  1.,  83., 202.,  76.,  27.], device='cuda:0')
5
0
['/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/map_sample']
['/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/val/map_sample']
/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/secondary_strctures
/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/val/secondary_strctures
/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/checkpoint/secondary_strctures/secondary_strctures.epoch0.pth.tar
/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/evaluation/secondary_strctures


In [12]:
# Generate data loaders
train_loader, val_loader = get_loaders(train_map_dirs, train_model_dir,
                                       val_map_dirs, val_model_dir,
                                       batch_size, num_workers,
                                       pin_memory)

In [13]:
print(train_loader)
print(val_loader)

In [14]:
data, targets = next(iter(train_loader))
data, targets = data.to(device=DEVICE), targets.long().to(device=DEVICE)

print(data.shape)
print(targets.shape)


torch.Size([1, 1, 64, 64, 64])
torch.Size([1, 64, 64, 64])


In [ ]:
with torch.cuda.amp.autocast():
    predictions = model(data)
    print(data.shape)
    print(predictions.shape)
    print(targets.shape)

    loss = loss_fn(predictions, targets)


In [ ]:
import numpy as np

for num in [0, 5, 6, 8, 9, 10]:
    for model_type in ['secondary_strctures', 'key_atoms', 'residue_types']:
        model_type = 'residue_types'
        filename = f'/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/{model_type}/model_{model_type}.{num}.npy'
        aaa = np.load(filename)

        print(np.max(aaa))
    print("###")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for num in range(0,5):
    model_type = 'map_sample'
    mapname = f'/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/{model_type}/map.{num}.npy'
    aaa = np.load(mapname)
    # print(np.max(aaa))
    plt.imshow(aaa[:, :, 30], cmap='gray', origin='lower')
    plt.title(f'{model_type}.{num}')
    plt.show()
    for model_type in ['secondary_strctures', 'key_atoms', 'residue_types']:
        filename = f'/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/{model_type}/model_{model_type}.{num}.npy'
        aaa = np.load(filename)
        plt.imshow(aaa[:, :, 30], origin='lower')
        plt.title(f'{model_type}.{num}')
        plt.show()
    print("###")
